# 1、SummarizationMiddleware中间件


这种方式对于简单场景已经足够，但一旦进入真实项目，往往会遇到很多额外需求，例如：

想根据问题复杂度动态 切换模型 ；

想 限制 某些用户只能调用部分工具；

想在工具报错时 自动重试 或返回兜底结果；

想在模型调用前 插入额外的系统提示 ；

想记录每一步的 执行日志 ，方便排查问题；

想在敏感信息出现时 阻断执行 ；

想在正式执行工具前增加 人工审批 。

这些需求有一个共同特点：它们不是 Agent 的核心业务逻辑，但又会影响 Agent 的执行过程。如果把这些逻辑全部直接写进主流程，会带来几个问题：

## 1）主流程会迅速变乱
Agent 本身只需要关心“理解用户需求、决定是否调用工具、生成结果”，

但一旦把日志、鉴权、重试、风控、审计都塞进去，主逻辑就会变得臃肿。

## 2）很多逻辑是横切需求，难以复用

例如日志、重试、风控、权限控制，通常不是某一个 Agent 独有的，而是多个 Agent 都需要。如果直接
写死在每个 Agent 里，会产生大量重复代码。

## 3）流程控制粒度不够细

有些逻辑必须发生在“模型调用前”，有些要发生在“工具调用后”，如果没有统一的执行拦截点，开发者只
能手动改主流程，既麻烦又容易出错。

## 4）后期维护成本高

当你需要增加一个新规则，例如“所有外部工具调用前都先做审计”，如果系统没有中间件机制，往往需
要修改很多处代码。

## 总结：

中间件的价值就在于把这些与业务无关、但与执行过程强相关的横切逻辑，从 Agent 主流程中分离出
来。让Agent 主体代码 聚焦业务 ，而借助中间件，实现“ 拦截流程、修改流程、增强流程 ”。

简言之，LangChain 1.x 的中间件能实现如下功能：

✅ 日志与分析 - 追踪行为、调试、性能监控

✅ 转换 - 修改提示词、工具选择、输出格式

✅ 容错 - 重试、降级、早期终止

✅ 安* - 限流、守护规则、PII 检


## 举例1：测试trigger、keep参数

In [1]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [3]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
Carry out a simple friendly introduction in Chinese between the user (“老王”) and the assistant.

## SUMMARY
The conversation was a brief greeting exchange:
- The user said hello and asked who the assistant is.
- The assistant replied: “你好老王，我是小王.”
- The user responded that it was nice to meet the assistant.
No task beyond casual introduction was established.

## ARTIFACTS
None

## NEXT STEPS
Continue the conversation politely if the user sends another message.
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

我是开个玩笑啦，意思是“别急着太高兴”，有点调侃的语气。

如果你愿意，我可以继续用轻松一点的方式跟你聊。


## 举例2：summary_prompt

In [4]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

对历史消息的摘要如下：

用户先打招呼并自我介绍为“老王”，询问助手是谁；助手回答自己是“小王”；随后用户表示很高兴认识“小王”。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王别急，我这是自嘲式玩笑：意思是“先别急着高兴，我这人可能没你想的那么靠谱”，不是针对你，也没有恶意。要是让你不舒服了，我收回，改成：我也很高兴认识你，老王。
